# 🔄 Incremental Source Data Generator

**Shadowfax Logistics, `sl_ingest` incremental feed**

The incremental counterpart to `environment_setup.ipynb`: where setup does a one-time `overwrite` seed, this appends a **fresh batch** to the `sl_ingest` sources on every run and can inject schema drift, bad records, duplicates and late data. Run it before each pipeline execution so the DLT and PySpark tracks always process new, potentially messy data.

**Run-tracking table.** Every run appends to `_generator_runs`, per source: rows added, counts of bad/duplicate/late records, the drift action, the **cumulative active extra columns**, and a `classification`. At the start of each run the notebook rebuilds the persistent drift state from that table, so once a column drifts in it stays in.

Parameters are read like `setup_job.yml` (`env = ${var.env}`), so it drops into a job task unchanged. Generation logic uses plain Python loops + explicit schemas, mirroring the setup notebook, easy to tweak.

In [0]:
# NOTE: This data-generation notebook is co-authored and maintained by Claude.
# Per the repo convention the setup / seed notebooks are Claude-owned (user reviews, refines structure,
# and optimizes); the pipeline and transformation code is written by the user.

from pyspark.sql.types import *
from pyspark.sql.functions import lit, col, row_number, rand
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import random

## Configuration

Row counts come from job parameters (override per run).

### What each dataset gets this run

Two kinds of mess, controlled separately:

1. **Probabilistic anomalies** (`bad`, `dupes`, `late`): turned on per dataset via its `inject` set. When on, they hit a small random share of rows (e.g. ~3% of orders get a missing amount). Left to chance.
2. **Schema drift**: a structural change, so you pick it explicitly per table in `DRIFT_THIS_RUN`. Each listed JSON table gets its next new column this run (deterministic), and that column persists on later runs. Defaults to empty, so drift only happens when you ask for it.
e.g. ["telemetry"] or ["shipment_events", "orders"]. Empty = no drift this run.

Schema drift is JSON-only; the Delta sources (`orders`, `customers`) keep their schema for this scenario and simplicity, so they are never drifted.

In [ ]:
# Job parameters arrive via getCurrentBindings(), same pattern as environment_setup.ipynb
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())

ENV         = configs.get("env", "dev")
SCHEMA_INGEST = ENV
LANDING_BASE  = f"/Volumes/sl_ingest/{SCHEMA_INGEST}/landing"

# Run-tracking table location
AUDIT_TABLE = f"sl_ingest.{SCHEMA_INGEST}._incremental_generator_runs"

# A new seed every run so each batch differs; BATCH_ID makes appended files/ids unique
BATCH_TS = datetime.now()
BATCH_ID = BATCH_TS.strftime("%Y%m%d%H%M%S")
random.seed(int(BATCH_TS.timestamp()))  # reproducible randomness per batch

N_ORDERS    = int(configs.get("n_orders",    "500"))   # NEW orders this batch (INSERT into the orders CDC feed)
N_CUSTOMERS = int(configs.get("n_customers", "20"))    # customer rows this batch (change + new + churn)
N_EVENTS    = int(configs.get("n_events",    "5000"))  # new shipment events this batch
N_TELEMETRY = int(configs.get("n_telemetry", "2000"))  # new telemetry readings this batch
N_CDC       = int(configs.get("n_cdc",       "275"))   # status UPDATEs to existing orders this batch
N_VEH_UPD   = int(configs.get("n_vehicle_updates", "2"))  # rare fleet-master depot reassignments (SCD1, full-row post-images)

INJECT_ANOMALIES = configs.get("inject_anomalies", True)
DRIFT_THIS_RUN = configs.get("drift_this_run", "").split(',')


In [ ]:
JSON_SOURCES = {"orders", "shipment_events", "telemetry", "vehicles"}

# 1) Probabilistic anomalies per dataset: a small random share of rows when listed.
#    Options: "bad", "dupes", "late".  (Drift is handled separately below.)
DATASETS = {
    "orders":          {"count": N_ORDERS,    "inject": {"bad"}},   # file CDC: INSERT + UPDATE drops
    "customers":       {"count": N_CUSTOMERS, "inject": set()},     # Delta CDF: change + new + churn
    "shipment_events": {"count": N_EVENTS,    "inject": {"bad", "dupes", "late"}},
    "telemetry":       {"count": N_TELEMETRY, "inject": {"bad"}},
    "vehicles":        {"count": N_VEH_UPD,   "inject": set()},     # fleet master: SCD1 full-row post-images
}

def chance(p):
    """True with probability p (one random draw)."""
    return random.random() < p

def wants(name, anomaly):
    if anomaly == "drift":
        return name in DRIFT_THIS_RUN and name in JSON_SOURCES
    return anomaly in DATASETS[name]["inject"]

print(f"env={ENV}  counts:", {k: v["count"] for k, v in DATASETS.items()})
for k, v in DATASETS.items():
    print(f"  {k:16s} anomalies: {sorted(v['inject']) or 'none'}")
print(f"  drift this run: {DRIFT_THIS_RUN or 'none'}")


In [0]:
spark.sql("USE CATALOG sl_ingest")
print(f"Appending batch {BATCH_ID} to sl_ingest / {LANDING_BASE}")

## Run-tracking table & persistent schema state

The audit table is append-only history. On each run we rebuild `drift_state` = the latest `active_extra_columns` per source, so columns that drifted in on earlier runs keep being emitted. This is what lets the next run "know" a column was permanently added.

In [0]:
# The run-tracking table: append-only history, one row per (batch, source).
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
        batch_id              STRING,
        run_ts                TIMESTAMP,
        env                   STRING,
        source                STRING,
        rows_added            INT,
        bad_records           INT,
        duplicates            INT,
        late_records          INT,
        drift_action          STRING,
        active_extra_columns  STRING,
        classification        STRING
    )
""")

# Rebuild the persistent drift state. For each source, take its most recent audit row
# and read back the columns that have drifted in so far. This is what lets a column that
# appeared on an earlier run keep being emitted this run.
drift_state = {}
if spark.table(AUDIT_TABLE).head(1):                          # table already has rows
    newest_first = Window.partitionBy("source").orderBy(
        col("run_ts").desc(), col("batch_id").desc())
    latest_rows = (
        spark.table(AUDIT_TABLE)
        .withColumn("rn", row_number().over(newest_first))
        .filter("rn = 1")                                     # keep only the newest row per source
        .select("source", "active_extra_columns")
        .collect()
    )
    for r in latest_rows:
        columns = [c for c in (r["active_extra_columns"] or "").split(",") if c]
        if columns:
            drift_state[r["source"]] = columns

print(f"Audit table: {AUDIT_TABLE}")
print(f"Carried-forward drift state: {drift_state or 'none yet'}")


def classify(bad, dup, late, drift_action):
    """Turn the per-source counts into a single label for the audit row."""
    tags = []
    if drift_action and drift_action != "none":
        tags.append("schema_drift")
    if bad > 0:
        tags.append("bad_records")
    if dup > 0:
        tags.append("duplicates")
    if late > 0:
        tags.append("late_data")
    return ",".join(tags) if tags else "healthy"

## Schema-drift helper (deterministic, JSON-only)

When `drift` is enabled for a JSON dataset, each run adds the **next** new candidate column (until the list is exhausted) and re-applies every column added before, so drift accumulates exactly like a real upstream schema change. Delta sources are never drifted.

In [ ]:
DRIFT_CANDIDATES = {
    "shipment_events": ["device_firmware", "gps_accuracy_m", "carrier_code"],
    "telemetry":       ["tire_pressure_psi", "battery_voltage", "ambient_temp_c"],
    "orders":          ["source_batch", "ingest_region"],
}

def _drift_value(c):
    if c == "device_firmware":
        return f"fw{random.randint(1, 9)}.{random.randint(0, 9)}"
    if c in ("gps_accuracy_m", "tire_pressure_psi", "battery_voltage", "ambient_temp_c"):
        return str(round(random.uniform(1, 40), 1))
    if c == "source_batch":
        return BATCH_ID
    if c == "ingest_region":
        return random.choice(["CH-N", "CH-S", "CH-E", "CH-W"])
    if c == "carrier_code":
        return random.choice(["SLX", "PSTCH", "DHLCH"])
    return "drift"

def maybe_drift(df, name):
    active = list(drift_state.get(name, []))               # columns added on earlier runs (persist)
    action = "none"
    if wants(name, "drift"):
        nxt = next((c for c in DRIFT_CANDIDATES.get(name, []) if c not in active), None)
        if nxt:
            active.append(nxt)                             # deterministic: add the next new column
            action = f"added column '{nxt}'"
    for c in active:
        df = df.withColumn(c, lit(_drift_value(c)))
    drift_state[name] = active
    return df, action, ",".join(active)


## Reference data

The same lookups used by the initial seed, so generated records stay consistent with the existing dataset.

In [0]:
me_cities = ["Minas Tirith", "Osgiliath", "Pelargir", "Dol Amroth", "Edoras",
             "Helm's Deep", "Aldburg", "Hobbiton", "Bywater", "Michel Delving",
             "Bree", "Rivendell", "Grey Havens", "Tharbad", "Esgaroth", "Dale",
             "Erebor", "Caras Galadhon"]
tiers = ["standard", "premium", "enterprise"]
industries = ["pharma", "retail", "manufacturing", "food_beverage", "tech", "finance"]

# Customer names: canonical Middle-earth characters first (The Lord of the Rings,
# The Hobbit, The Silmarillion), then generated Shire-style names once canon runs out.
me_characters = [
    # Hobbits of the Shire (and thereabouts)
    "Frodo Baggins", "Bilbo Baggins", "Samwise Gamgee", "Meriadoc Brandybuck",
    "Peregrin Took", "Rosie Cotton", "Hamfast Gamgee", "Lobelia Sackville-Baggins",
    "Otho Sackville-Baggins", "Lotho Sackville-Baggins", "Fredegar Bolger",
    "Folco Boffin", "Drogo Baggins", "Primula Brandybuck", "Paladin Took",
    "Eglantine Banks", "Esmeralda Brandybuck", "Saradoc Brandybuck", "Gerontius Took",
    "Bandobras Took", "Belladonna Took", "Bungo Baggins", "Elanor Gamgee",
    "Farmer Maggot", "Ted Sandyman", "Will Whitfoot", "Robin Smallburrow",
    "Tolman Cotton", "Holman Greenhand", "Halfast Gamgee", "Angelica Baggins",
    "Milo Burrows", "Dora Baggins", "Adelard Took", "Everard Took",
    "Melilot Brandybuck", "Celandine Brandybuck", "Mentha Brandybuck",
    "Pervinca Took", "Pearl Took", "Pimpernel Took", "Diamond of Long Cleeve",
    "Estella Bolger", "Déagol", "Sméagol", "Marcho Fallohide", "Blanco Fallohide",
    "Bucca of the Marish",
    # Bree-land and stranger folk
    "Barliman Butterbur", "Bill Ferny", "Harry Goatleaf", "Tom Bombadil", "Goldberry",
    # Men of Gondor, Rohan, Dale and the North
    "Aragorn Elessar", "Boromir", "Faramir", "Denethor II", "Théoden", "Éomer",
    "Éowyn", "Gríma Wormtongue", "Bard the Bowman", "Beorn", "Théodred",
    "Erkenbrand", "Gamling", "Háma", "Elfhelm", "Grimbold", "Dúnhere",
    "Imrahil of Dol Amroth", "Forlong the Fat", "Húrin of the Keys", "Beregond",
    "Bergil", "Ioreth", "Ingold", "Damrod", "Mablung of Ithilien", "Anborn",
    "Halbarad", "Arathorn", "Ecthelion II", "Isildur", "Anárion", "Elendil",
    "Meneldil", "Cirion", "Eorl the Young", "Helm Hammerhand", "Fréaláf Hildeson",
    "Brytta Léofa", "Walda", "Folca", "Fengel", "Thengel", "Morwen of Lossarnach",
    "Girion of Dale", "Brand of Dale", "Bain of Dale", "Éothain", "Céorl",
    "Targon", "Golasgil", "Hirluin the Fair", "Duinhir", "Dervorin", "Angbor",
    # Elves
    "Legolas", "Galadriel", "Celeborn", "Elrond", "Arwen Undómiel", "Glorfindel",
    "Erestor", "Lindir", "Haldir", "Rúmil", "Orophin", "Thranduil", "Círdan",
    "Gildor Inglorion", "Celebrían", "Elladan", "Elrohir", "Fëanor", "Fingolfin",
    "Finarfin", "Fingon", "Turgon", "Finrod Felagund", "Angrod", "Aegnor",
    "Maedhros", "Maglor", "Celegorm", "Caranthir", "Curufin", "Amrod", "Amras",
    "Celebrimbor", "Gil-galad", "Elu Thingol", "Lúthien Tinúviel", "Beleg Cúthalion",
    "Mablung of Doriath", "Daeron", "Eöl", "Maeglin", "Aredhel", "Idril Celebrindal",
    "Voronwë", "Ecthelion of the Fountain", "Egalmoth", "Duilin", "Galdor",
    "Nimrodel", "Amroth", "Finduilas", "Orodreth", "Gwindor", "Saeros", "Tauriel",
    # Dwarves
    "Thorin Oakenshield", "Balin", "Dwalin", "Fíli", "Kíli", "Dori", "Nori", "Ori",
    "Óin", "Glóin", "Bifur", "Bofur", "Bombur", "Gimli", "Dáin Ironfoot",
    "Thráin II", "Thrór", "Durin the Deathless", "Telchar", "Mîm", "Azaghâl",
    "Náin", "Frerin", "Dís", "Fundin", "Gróin", "Borin", "Farin",
    # Wizards, Ainur and other powers
    "Gandalf the Grey", "Saruman the White", "Radagast the Brown", "Alatar",
    "Pallando", "Melian", "Ossë", "Uinen", "Eönwë", "Ilmarë", "Arien", "Tilion",
    "Manwë", "Varda Elentári", "Ulmo", "Aulë", "Yavanna", "Oromë", "Vána",
    "Námo Mandos", "Vairë", "Irmo Lórien", "Estë", "Nienna", "Tulkas", "Nessa",
    # Men of the First Age
    "Beren Erchamion", "Barahir", "Húrin Thalion", "Huor", "Morwen Eledhwen",
    "Rían", "Túrin Turambar", "Niënor Níniel", "Tuor", "Eärendil", "Elwing",
    "Hador Lórindol", "Bëor the Old", "Haleth", "Brandir", "Dorlas", "Emeldir",
    "Aerin", "Ulfang", "Bór", "Amlach", "Marach",
    # Ents and other notable clients
    "Treebeard", "Quickbeam", "Leaflock", "Skinbark", "Ghân-buri-Ghân", "Smaug",
]
me_first_names = ["Tolman", "Wilcome", "Hobson", "Andwise", "Anson", "Hamson",
                  "Halfred", "Daisy", "May", "Marigold", "Hilda", "Mirabella",
                  "Donnamira", "Isengrim", "Isumbras", "Ferumbras", "Fortinbras",
                  "Flambard", "Sigismond", "Hildigrim", "Adalgrim", "Reginard",
                  "Odovacar", "Rudigar", "Herugar", "Madoc", "Marmadoc", "Gorbadoc",
                  "Rorimac", "Dinodas", "Marmadas", "Seredic", "Léofwine", "Éadric",
                  "Fréawine", "Goldwine", "Déor", "Gram", "Baldor", "Aldor"]
me_surnames = ["Bracegirdle", "Proudfoot", "Hornblower", "Goodbody", "Brockhouse",
               "Chubb", "Grubb", "Burrows", "Sandheaver", "Tunnelly", "Underhill",
               "Greenhand", "Whitfoot", "Noakes", "Rumble", "Twofoot", "Longholes",
               "Banks", "Brownlock", "Puddifoot", "Oldbuck", "Boffin", "Bolger",
               "Goodchild", "Cotton", "Gamwich", "Roper", "Gardner", "of Bree",
               "of Dale", "of Esgaroth", "of the Mark", "of Lossarnach"]

def character_for(cid):
    if cid <= len(me_characters):
        return me_characters[cid - 1]
    return f"{random.choice(me_first_names)} {random.choice(me_surnames)}"

def email_for(name, cid):
    import unicodedata
    ascii_name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    slug = "".join(ch if ch.isalnum() else "." for ch in ascii_name.lower().replace("'", ""))
    slug = ".".join(p for p in slug.split(".") if p)
    return f"{slug}.{cid}@ravenpost.me"

products = [("PKG_STD", "Standard Parcel", 12.50), ("PKG_EXP", "Express Parcel", 24.90),
            ("PKG_SAME", "Same-Day Delivery", 39.90), ("FRT_PAL", "Pallet Freight", 89.00),
            ("FRT_FTL", "Full Truck Load", 450.00), ("COLD", "Cold Chain Parcel", 34.90),
            ("DOC", "Document Courier", 8.90), ("INTL", "International Shipment", 65.00)]
statuses = ["created", "confirmed", "picked_up", "in_transit", "out_for_delivery",
            "delivered", "failed_delivery", "returned"]
payment_methods = ["invoice", "credit_card", "bank_transfer", "paypal"]
event_types = ["gps_ping", "status_change", "temperature_alert", "delay_alert", "signature_captured"]

## 1 · Orders: file-based CDC change feed (JSON append)

Orders are a **file-based CDC feed**, not a Delta table. Each batch appends a new change drop to the `orders` landing path: brand-new orders as **INSERT** (full rows) and status changes to existing orders as **UPDATE** (full row post-images: only `status` / `delivery_date` differ from the order's current values). No deletes in this feed by design. `_change_ts` (batch time) is the sequence column the SCD2 build orders versions by; the snapshot written by `environment_setup` sorts before every change here. Bad-record injection adds the wrinkles silver must catch: missing amounts, the occasional negative amount, and a blank destination city.

In [ ]:
# Highest existing ORD-###### across the whole orders feed (snapshot + prior inserts),
# so new ids continue the sequence (0 if the feed is somehow empty).
mx = (spark.read.json(f"{LANDING_BASE}/orders")
      .selectExpr("max(cast(substring(order_id, 5) as int)) m").collect()[0]["m"]) or 0

N_INSERTS = N_ORDERS   # brand-new orders (full rows)
N_UPDATES = N_CDC      # status changes to existing orders (full post-image UPDATEs)
order_bad = 0
orders_cdc = []

# INSERTs — brand-new orders continue the id sequence, full payload
for k in range(N_INSERTS):
    oid = f"ORD-{mx + k + 1:06d}"
    cid = random.randint(1, 200)
    prod = random.choice(products)
    qty = random.randint(1, 10) if prod[0].startswith("PKG") else random.randint(1, 3)
    odate = BATCH_TS - timedelta(minutes=random.randint(0, 180))
    amount = round(prod[2] * qty * random.uniform(0.9, 1.1), 2)
    origin, dest = random.choice(me_cities), random.choice(me_cities)
    if wants("orders", "bad"):
        if chance(0.03):
            amount = None          # missing amount
            order_bad += 1
        elif chance(0.01):
            amount = -abs(amount)  # negative amount
            order_bad += 1
        if chance(0.01):
            dest = ""              # blank destination
            order_bad += 1
    orders_cdc.append((oid, cid, prod[0], prod[1], qty, amount, "created",
                       random.choice(payment_methods), odate, None, origin, dest,
                       "INSERT", BATCH_TS))

# UPDATEs — status changes to existing orders, carrying the FULL current row (full post-image).
# Real DB-sourced CDC (Debezium, Delta CDF, logical replication) emits complete after-images, not
# changed-columns-only, so each UPDATE reuses the order's latest values and overrides only status
# (and delivery_date). Sampled without replacement: no order gets two same-_change_ts versions per batch.
current_orders = (spark.read.json(f"{LANDING_BASE}/orders")
                  .withColumn("_rn", row_number().over(
                      Window.partitionBy("order_id").orderBy(col("_change_ts").desc())))
                  .filter("_rn = 1")
                  .select("order_id", col("customer_id").cast("int"),
                          "product_code", "product_name", col("quantity").cast("int"),
                          col("total_amount").cast("double"), "payment_method",
                          col("order_date").cast("timestamp"), "origin_city", "destination_city"))
for r in current_orders.orderBy(rand()).limit(N_UPDATES).collect():
    new_status = random.choice(["confirmed", "picked_up", "in_transit",
                                "out_for_delivery", "delivered", "failed_delivery", "returned"])
    ddate = BATCH_TS if new_status in ("delivered", "returned") else None
    orders_cdc.append((r["order_id"], r["customer_id"], r["product_code"], r["product_name"],
                       r["quantity"], r["total_amount"], new_status, r["payment_method"],
                       r["order_date"], ddate, r["origin_city"], r["destination_city"],
                       "UPDATE", BATCH_TS))

orders_schema = StructType([
    StructField("order_id", StringType()), StructField("customer_id", IntegerType(), True),
    StructField("product_code", StringType(), True), StructField("product_name", StringType(), True),
    StructField("quantity", IntegerType(), True), StructField("total_amount", DoubleType(), True),
    StructField("status", StringType()), StructField("payment_method", StringType(), True),
    StructField("order_date", TimestampType(), True), StructField("delivery_date", TimestampType(), True),
    StructField("origin_city", StringType(), True), StructField("destination_city", StringType(), True),
    StructField("_change_type", StringType()), StructField("_change_ts", TimestampType()),
])

df_orders, drift_o, active_o = maybe_drift(
    spark.createDataFrame(orders_cdc, orders_schema), "orders")
df_orders.repartition(1).write.mode("append").json(f"{LANDING_BASE}/orders")
n_orders_added = len(orders_cdc)
print(f"  ✓ orders: +{N_INSERTS} INSERT, +{N_UPDATES} UPDATE ({n_orders_added} rows) -> {LANDING_BASE}/orders")


## 2 · Customers: operational changes (Postgres-style, captured via CDF)

`sl_ingest.customers` simulates the operational **Postgres** customers table: current
state only, one row per customer, no SCD2 columns. Each batch mutates it the way an
application would, and Delta Change Data Feed turns those mutations into the change
events the warehouse needs to build a Type 2 dimension:

- **Update**: a subset of existing customers change tier and address/city in place (`last_updated` set to the batch time). CDF emits `update_preimage` / `update_postimage`.
- **Delete**: a few customers churn and are removed. CDF emits `delete`.
- **Insert**: brand-new customers continue the id sequence. CDF emits `insert`.
- **Unchanged**: everyone not picked is left alone, so silver has to ignore no-ops.

The SCD2 history (`valid_from` / `valid_to` / `is_current`) is **not** stored here.
Silver reconstructs it from the bronze CDF log using `_change_type` and commit order.

In [0]:
# ── Customers: operational changes against the current-state source ──
# sl_ingest.customers simulates a Postgres "customers" table: one current-state
# row per customer, no SCD2 columns. We mutate it the way an app would, so the
# Change Data Feed emits real insert / update / delete events that the warehouse
# turns into a Type 2 dimension:
#   1. UPDATE    a subset change tier + address/city in place (last_updated=now).
#                CDF -> update_preimage / update_postimage.
#   2. DELETE    a few customers churn and are removed.  CDF -> delete.
#   3. INSERT    brand-new customers continue the id sequence.  CDF -> insert.
#   4. UNCHANGED everyone not picked is left alone.
from delta.tables import DeltaTable

source_tbl = f"sl_ingest.{SCHEMA_INGEST}.customers"
streets = ["Bagshot Row", "Great East Road", "The Greenway", "Old South Road", "Market Square", "Harbour Road"]

existing_ids = [r["customer_id"] for r in spark.table(source_tbl).select("customer_id").collect()]
mxc = max(existing_ids) if existing_ids else 0

n_update = max(1, round(N_CUSTOMERS * 0.55))   # existing customers changed in place
n_delete = max(0, round(N_CUSTOMERS * 0.10))   # existing customers deleted (churn)
n_new    = max(1, round(N_CUSTOMERS * 0.35))   # brand-new customers

random.shuffle(existing_ids)
update_ids = existing_ids[:n_update]
delete_ids = existing_ids[n_update:n_update + n_delete]

dt = DeltaTable.forName(spark, source_tbl)

# 1) In-place UPDATEs, applied as one batch via MERGE (CDF logs pre/post images).
update_recs = [(cid, random.choice(tiers),
                f"{random.randint(1, 200)} {random.choice(streets)}",
                random.choice(me_cities), f"{random.randint(1000, 9999)}")
               for cid in update_ids]
if update_recs:
    upd_schema = StructType([
        StructField("customer_id", IntegerType()), StructField("tier", StringType()),
        StructField("address", StringType()), StructField("city", StringType()),
        StructField("postal_code", StringType())])
    updates_df = spark.createDataFrame(update_recs, upd_schema)
    (dt.alias("t").merge(updates_df.alias("u"), "t.customer_id = u.customer_id")
       .whenMatchedUpdate(set={"tier": "u.tier", "address": "u.address", "city": "u.city",
                               "postal_code": "u.postal_code", "last_updated": lit(BATCH_TS)})
       .execute())

# 2) DELETEs: churned customers are removed from the source.
if delete_ids:
    dt.delete(col("customer_id").isin(delete_ids))

# 3) INSERTs: brand-new customers continue the id sequence.
new_recs = []
for j in range(n_new):
    cid = mxc + j + 1
    name = character_for(cid)
    new_recs.append((cid, name, email_for(name, cid),
                     f"{random.randint(1, 200)} Great East Road", random.choice(me_cities),
                     f"{random.randint(1000, 9999)}", random.choice(tiers), random.choice(industries),
                     BATCH_TS, BATCH_TS))
cust_schema = StructType([
    StructField("customer_id", IntegerType()), StructField("company_name", StringType()),
    StructField("contact_email", StringType()), StructField("address", StringType()),
    StructField("city", StringType()), StructField("postal_code", StringType()),
    StructField("tier", StringType()), StructField("industry", StringType()),
    StructField("signup_date", TimestampType()), StructField("last_updated", TimestampType())])
if new_recs:
    (spark.createDataFrame(new_recs, cust_schema).write.mode("append")
     .option("delta.enableChangeDataFeed", "true").saveAsTable(source_tbl))

n_cust_added = len(new_recs)
print(f"  ✓ customers: {len(update_ids)} updated, {len(delete_ids)} deleted, {n_new} new")

## 3 · Shipment events: JSON append

High-volume tracking events written as **new JSON files** (Auto Loader reads them as an incremental batch). When enabled: ~3% duplicate event IDs and ~4% late / out-of-order timestamps; bad records null the vehicle ID or push temperatures out of range. Persistent schema drift may add a column for this and all future batches.

In [0]:
max_order_no = mx + N_ORDERS  # highest order id after this batch's orders were added
events_bad = events_dupes = events_late = 0
event_recs = []
for i in range(N_EVENTS):
    eid = f"EVT-{BATCH_ID}-{i:06d}"
    oid = f"ORD-{random.randint(1, max_order_no):06d}"
    veh = f"VH-{random.randint(1, 200):03d}"
    et = random.choices(event_types, weights=[60, 20, 5, 10, 5])[0]
    ts = BATCH_TS - timedelta(minutes=random.randint(0, 240))
    if wants("shipment_events", "late") and chance(0.04):
        ts = BATCH_TS - timedelta(days=random.randint(3, 30))  # late / out-of-order
        events_late += 1
    lat = round(random.uniform(46.0, 47.8), 6) if et == "gps_ping" else None
    lon = round(random.uniform(6.0, 10.5), 6) if et == "gps_ping" else None
    st = random.choice(statuses) if et == "status_change" else None
    tmp = round(random.gauss(4, 2), 1) if et == "temperature_alert" else None
    if wants("shipment_events", "bad"):
        if chance(0.02):
            veh = None    # missing vehicle id
            events_bad += 1
        if tmp is not None and chance(0.05):
            tmp = 999.0   # out-of-range temperature
            events_bad += 1
    row = (eid, oid, veh, et, ts, lat, lon, st, tmp, random.choice(["kafka", "iot_hub", "api"]))
    event_recs.append(row)
    if wants("shipment_events", "dupes") and chance(0.03):
        event_recs.append(row)  # duplicate event
        events_dupes += 1

event_schema = StructType([
    StructField("event_id", StringType()), StructField("order_id", StringType()),
    StructField("vehicle_id", StringType(), True), StructField("event_type", StringType()),
    StructField("event_timestamp", TimestampType()),
    StructField("latitude", DoubleType(), True), StructField("longitude", DoubleType(), True),
    StructField("shipment_status", StringType(), True), StructField("temperature_celsius", DoubleType(), True),
    StructField("source_system", StringType())])

df_events, drift_e, active_e = maybe_drift(spark.createDataFrame(event_recs, event_schema), "shipment_events")
df_events.repartition(2).write.mode("append").json(f"{LANDING_BASE}/shipment_events")
n_events_added = len(event_recs)
print(f"  ✓ shipment_events: +{n_events_added} json records | bad: {events_bad} dupes: {events_dupes} late: {events_late} | drift: {drift_e}")

## 4 · Vehicle telemetry: JSON append

Sensor readings keeping the same 60%-to-5-vehicles **skew** as the initial seed so partition/clustering optimisation stays relevant. Bad records inject impossible speeds; persistent drift may add a column.

In [0]:
import uuid
tel_bad = 0
tel_recs = []
for i in range(N_TELEMETRY):
    veh = f"VH-{random.randint(1, 5):03d}" if random.random() < 0.6 else f"VH-{random.randint(6, 200):03d}"
    ts = BATCH_TS - timedelta(minutes=random.randint(0, 240), seconds=random.randint(0, 59))
    speed = round(random.uniform(0, 120), 1)
    if wants("telemetry", "bad") and chance(0.01):
        speed = round(random.uniform(300, 900), 1)  # impossible speed
        tel_bad += 1
    fuel = round(random.uniform(0, 100), 1)
    etemp = round(random.gauss(22, 8), 1)
    ctemp = round(random.gauss(4, 3), 1) if random.random() < 0.3 else None
    odo = random.randint(0, 500000)
    tel_recs.append((str(uuid.uuid4()), veh, ts, speed, fuel, etemp, ctemp, odo,
                     random.choice(["idle", "moving", "loading", "maintenance"])))

tel_schema = StructType([
    StructField("reading_id", StringType()),
    StructField("vehicle_id", StringType()), StructField("reading_timestamp", TimestampType()),
    StructField("speed_kmh", DoubleType()), StructField("fuel_pct", DoubleType()),
    StructField("engine_temp_c", DoubleType()), StructField("cargo_temp_c", DoubleType(), True),
    StructField("odometer_km", IntegerType()), StructField("vehicle_status", StringType())])

df_tel, drift_t, active_t = maybe_drift(spark.createDataFrame(tel_recs, tel_schema), "telemetry")
df_tel.repartition(1).write.mode("append").json(f"{LANDING_BASE}/telemetry")
n_tel_added = len(tel_recs)
print(f"  ✓ telemetry: +{n_tel_added} json records | bad: {tel_bad} | drift: {drift_t}")

## 5 · Vehicles: fleet-master depot reassignments (JSON append)

The fleet master is a slow-moving **SCD1 reference source**. Rare depot reassignments are appended as **full-row post-images** to the same `vehicles/` landing path the seed wrote to, so one Auto Loader stream reads seed and updates alike, and downstream stays a plain SCD1 upsert (latest row per `vehicle_id` by `last_updated` wins). Deliberately simpler than orders: no snapshot backfill, no SCD2 history, no deletes.


In [ ]:
# ── Vehicles: rare depot reassignments (full-row post-images, SCD1) ──
depot_cities = ["Minas Tirith", "Edoras", "Hobbiton", "Bree", "Pelargir", "Dol Amroth", "Esgaroth", "Rivendell"]

n_veh_added, drift_v, active_v = 0, "none", ""
if N_VEH_UPD > 0:
    # current state = latest row per vehicle across seed + all prior update drops
    latest_per_vehicle = Window.partitionBy("vehicle_id").orderBy(col("last_updated").desc())
    veh_latest = (spark.read.json(f"{LANDING_BASE}/vehicles/")
                  .withColumn("rn", row_number().over(latest_per_vehicle))
                  .filter("rn = 1").drop("rn"))

    veh_recs = []
    for r in veh_latest.orderBy(rand()).limit(N_VEH_UPD).collect():
        d = r.asDict()
        d["home_depot"] = random.choice([c for c in depot_cities if c != d["home_depot"]])
        d["last_updated"] = BATCH_TS.isoformat()   # feed carries ISO strings; bronze casts
        veh_recs.append(d)

    df_veh = spark.createDataFrame(veh_recs, veh_latest.schema)
    df_veh, drift_v, active_v = maybe_drift(df_veh, "vehicles")
    df_veh.repartition(1).write.mode("append").json(f"{LANDING_BASE}/vehicles")
    n_veh_added = len(veh_recs)

print(f"  ✓ vehicles: +{n_veh_added} depot reassignments (full-row post-images)")


## 6 · Malformed lines (optional)

`df.write.json` only ever emits valid JSON, so to exercise `badRecordsPath` / `_rescued_data` / `PERMISSIVE` parsing we drop a small file of genuinely broken lines into the `shipment_events` landing dir. These count toward `shipment_events` bad records in the audit. Only runs when bad-record injection is on.

In [0]:
malformed_n = 0
if wants("shipment_events", "bad"):
    broken_lines = [
        '{"event_id":"EVT-BROKEN-1","order_id":"ORD-000001","vehicle_id":"VH-007"',           # truncated, no closing brace
        '{"event_id":"EVT-BROKEN-2","event_type":"gps_ping","temperature_celsius":"NaN-ish"}',  # wrong type
        'this is not json at all',                                                             # raw garbage
        '',                                                                                    # blank line
    ]
    malformed_path = f"{LANDING_BASE}/shipment_events/_malformed_{BATCH_ID}.json"
    dbutils.fs.put(malformed_path, "\n".join(broken_lines) + "\n", overwrite=True)
    malformed_n = len(broken_lines)
    events_bad += malformed_n
    print(f"  ✓ shipment_events: +{malformed_n} malformed lines -> {malformed_path}")
else:
    print("  (skipped malformed lines: 'bad' not enabled for shipment_events)")

## Write the run-tracking rows

One row per source for this batch, counts, drift action, the cumulative active columns, and the derived classification, appended to the audit table.

In [ ]:
audit_rows = [
    (BATCH_ID, BATCH_TS, ENV, "orders",          n_orders_added, order_bad,  0,            0,           drift_o, active_o, classify(order_bad, 0, 0, drift_o)),
    (BATCH_ID, BATCH_TS, ENV, "customers",        n_cust_added,   0,          0,            0,           "none",  "",       classify(0, 0, 0, "none")),
    (BATCH_ID, BATCH_TS, ENV, "shipment_events",  n_events_added, events_bad, events_dupes, events_late, drift_e, active_e, classify(events_bad, events_dupes, events_late, drift_e)),
    (BATCH_ID, BATCH_TS, ENV, "telemetry",        n_tel_added,    tel_bad,    0,            0,           drift_t, active_t, classify(tel_bad, 0, 0, drift_t)),
    (BATCH_ID, BATCH_TS, ENV, "vehicles",         n_veh_added,    0,          0,            0,           drift_v, active_v, classify(0, 0, 0, drift_v)),
]

audit_schema = StructType([
    StructField("batch_id", StringType()), StructField("run_ts", TimestampType()),
    StructField("env", StringType()), StructField("source", StringType()),
    StructField("rows_added", IntegerType()), StructField("bad_records", IntegerType()),
    StructField("duplicates", IntegerType()), StructField("late_records", IntegerType()),
    StructField("drift_action", StringType()), StructField("active_extra_columns", StringType()),
    StructField("classification", StringType())])

spark.createDataFrame(audit_rows, audit_schema).write.mode("append").saveAsTable(AUDIT_TABLE)


## Summary

In [ ]:
print("=" * 64)
print(f"Shadowfax Logistics, incremental batch {BATCH_ID} complete  (env={ENV})")
print("=" * 64)
for name in ["orders", "shipment_events", "telemetry", "vehicles"]:
    n_files = len(dbutils.fs.ls(f"{LANDING_BASE}/{name}"))
    print(f"  {LANDING_BASE}/{name}: {n_files} files total")
for t in ["customers"]:
    total = spark.table(f"sl_ingest.{SCHEMA_INGEST}.{t}").count()
    print(f"  sl_ingest.{SCHEMA_INGEST}.{t}: {total:,} rows total")
print()
print("This batch in the tracking table:")
display(spark.sql(f"SELECT source, rows_added, bad_records, duplicates, late_records, drift_action, active_extra_columns, classification FROM {AUDIT_TABLE} WHERE batch_id = '{BATCH_ID}' ORDER BY source"))
